# TinyLLM End-to-End Kaggle Notebook

In [ ]:
!git clone <YOUR_TINYLLM_REPO_URL> TinyLLM
%cd TinyLLM
!pip install -q -r requirements.txt || true

In [ ]:
!python prepare_data.py

In [ ]:
from pathlib import Path
p=Path("train.py")
txt=p.read_text()
if "import argparse" not in txt:
    txt=txt.replace("from pathlib import Path","from pathlib import Path\\nimport argparse")
if "parser = argparse.ArgumentParser()" not in txt:
    ins="""parser = argparse.ArgumentParser()
parser.add_argument("--pos",type=str,default="rope",choices=["rope","learned","sinusoidal"])
args=parser.parse_args()

"""
    txt=ins+txt
for s in ['"pos_encoding": "rope",','"pos_encoding": "learned",','"pos_encoding": "sinusoidal",']:
    txt=txt.replace(s,'"pos_encoding": args.pos,')
p.write_text(txt)
print("Patched train.py")

In [ ]:
import os, subprocess, shutil
os.makedirs("checkpoints",exist_ok=True)
for exp in ["rope","learned","sinusoidal"]:
    for f in ["best.pt","latest.pt"]:
        fp=os.path.join("checkpoints",f)
        if os.path.exists(fp): os.remove(fp)
    subprocess.run(["python","train.py","--pos",exp],check=True)
    shutil.copy("checkpoints/best.pt",f"checkpoints/{exp}_best.pt")
    shutil.copy("checkpoints/latest.pt",f"checkpoints/{exp}_latest.pt")
print("Done")

In [ ]:
import shutil
for exp in ["rope","learned","sinusoidal"]:
    shutil.make_archive(f"{exp}_checkpoints","zip","checkpoints",f"{exp}_best.pt")
print("ZIPs created")